# Chapter 08 — Regularization and Cross-Validation

Notebook 02 showed that high-degree polynomials overfit badly. Here we introduce
the tools to fight overfitting systematically: **regularization** (Ridge, Lasso,
ElasticNet) and **cross-validation** for principled model selection.

---
## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, learning_curve
)
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

rng = np.random.default_rng(42)

---
## 2. Why Regularization?

Ordinary Least Squares (OLS) minimises only the **training error**. When the model
has too many features or too much flexibility, OLS will:

- Assign large coefficients to exploit noise in the training set.
- Produce wildly different predictions on new data.

**Regularization** adds a **penalty term** to the cost function that discourages
large coefficients:

$$\text{Cost} = \underbrace{\sum (y_i - \hat{y}_i)^2}_{\text{RSS (fit the data)}} + \underbrace{\lambda \cdot \text{Penalty}(\boldsymbol{\beta})}_{\text{keep coefficients small}}$$

The hyperparameter $\lambda$ (called `alpha` in sklearn) controls the trade-off:

- $\lambda = 0$: no penalty — same as OLS.
- $\lambda \to \infty$: coefficients are forced toward zero — underfitting.

---
## 3. Generating the Overfitting Dataset

We use a cubic relationship with some noise, then create high-degree polynomial
features to give the model an opportunity to overfit.

In [ ]:
n = 80
x = rng.uniform(-3, 3, size=n)
y = 0.5 * x**3 - x**2 + 2 * x + 5 + rng.normal(0, 3, size=n)

X = x.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

plt.scatter(x, y, alpha=0.5, edgecolors='k', linewidths=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Data (True Relationship: Cubic)')
plt.show()

---
## 4. The Overfitting Baseline

Let's fit a degree-10 polynomial with plain OLS — this will overfit.

In [ ]:
pipe_ols = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])
pipe_ols.fit(X_train, y_train)

print(f'OLS (degree 10):')
print(f'  Train R²: {pipe_ols.score(X_train, y_train):.4f}')
print(f'  Test R²:  {pipe_ols.score(X_test, y_test):.4f}')
print(f'  Max |coef|: {np.abs(pipe_ols.named_steps["reg"].coef_).max():.2f}')

Notice the gap between train and test R², and the large coefficient magnitudes.
This is the problem regularization addresses.

---
## 5. Ridge Regression (L2 Penalty)

### 5.1 The Math

Ridge adds the **L2 penalty** — the sum of squared coefficients:

$$\text{Cost}_{\text{Ridge}} = \sum (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \beta_j^2$$

**Effect:** all coefficients are **shrunk toward zero**, but none are set exactly
to zero. The model retains all features but reduces their influence.

**Intuition:** imagine pulling all the coefficients toward zero with a rubber band.
The stronger the pull ($\alpha$), the closer they get to zero.

### 5.2 Ridge in Practice

In [ ]:
pipe_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0))
])
pipe_ridge.fit(X_train, y_train)

print(f'Ridge (alpha=1.0, degree 10):')
print(f'  Train R²: {pipe_ridge.score(X_train, y_train):.4f}')
print(f'  Test R²:  {pipe_ridge.score(X_test, y_test):.4f}')
print(f'  Max |coef|: {np.abs(pipe_ridge.named_steps["reg"].coef_).max():.2f}')

### 5.3 Ridge Coefficient Path

How do coefficients change as alpha increases?

In [ ]:
alphas = np.logspace(-3, 5, 100)

# We need to transform and scale the training data once
poly_tf = PolynomialFeatures(degree=10, include_bias=False)
scaler_tf = StandardScaler()
X_train_tf = scaler_tf.fit_transform(poly_tf.fit_transform(X_train))
X_test_tf = scaler_tf.transform(poly_tf.transform(X_test))

ridge_coefs = []
ridge_train_r2 = []
ridge_test_r2 = []

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_tf, y_train)
    ridge_coefs.append(ridge.coef_)
    ridge_train_r2.append(ridge.score(X_train_tf, y_train))
    ridge_test_r2.append(ridge.score(X_test_tf, y_test))

ridge_coefs = np.array(ridge_coefs)

plt.figure(figsize=(10, 5))
for i in range(ridge_coefs.shape[1]):
    plt.plot(alphas, ridge_coefs[:, i], linewidth=1)

plt.xscale('log')
plt.xlabel(r'$\alpha$ (regularization strength)')
plt.ylabel('Coefficient value')
plt.title('Ridge Coefficient Path — Coefficients Shrink Toward Zero')
plt.axhline(0, color='k', linewidth=0.5)
plt.show()

As $\alpha$ increases (left to right), all coefficients are squeezed toward zero.
None of them reach exactly zero — that is a key property of L2 regularization.

---
## 6. Lasso Regression (L1 Penalty)

### 6.1 The Math

Lasso uses the **L1 penalty** — the sum of absolute values of coefficients:

$$\text{Cost}_{\text{Lasso}} = \sum (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\beta_j|$$

**Effect:** Lasso can drive coefficients **exactly to zero**, effectively performing
**feature selection**. Irrelevant features are eliminated from the model.

**Intuition:** the L1 penalty has sharp corners at zero (think of a diamond shape
in 2-D). The optimum is more likely to land on a corner, where one or more
coefficients are exactly zero.

### 6.2 Lasso in Practice

In [ ]:
pipe_lasso = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Lasso(alpha=1.0, max_iter=10000))
])
pipe_lasso.fit(X_train, y_train)

lasso_coefs = pipe_lasso.named_steps['reg'].coef_
n_zero = np.sum(np.abs(lasso_coefs) < 1e-10)

print(f'Lasso (alpha=1.0, degree 10):')
print(f'  Train R²: {pipe_lasso.score(X_train, y_train):.4f}')
print(f'  Test R²:  {pipe_lasso.score(X_test, y_test):.4f}')
print(f'  Coefficients zeroed out: {n_zero} / {len(lasso_coefs)}')
print(f'  Non-zero coefs: {lasso_coefs[np.abs(lasso_coefs) > 1e-10].round(4)}')

Lasso has automatically eliminated most of the polynomial features, keeping only
the ones it considers important. This is **automatic feature selection**.

### 6.3 Lasso Coefficient Path

In [ ]:
lasso_coefs_path = []
lasso_test_r2 = []

for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(X_train_tf, y_train)
    lasso_coefs_path.append(lasso.coef_)
    lasso_test_r2.append(lasso.score(X_test_tf, y_test))

lasso_coefs_path = np.array(lasso_coefs_path)

plt.figure(figsize=(10, 5))
for i in range(lasso_coefs_path.shape[1]):
    plt.plot(alphas, lasso_coefs_path[:, i], linewidth=1)

plt.xscale('log')
plt.xlabel(r'$\alpha$ (regularization strength)')
plt.ylabel('Coefficient value')
plt.title('Lasso Coefficient Path — Coefficients Hit Zero')
plt.axhline(0, color='k', linewidth=0.5)
plt.show()

Unlike Ridge, Lasso coefficients actually hit zero and stay there as $\alpha$
increases. This is the visual signature of L1 sparsity.

---
## 7. ElasticNet (L1 + L2 Combined)

### 7.1 The Math

ElasticNet combines both penalties:

$$\text{Cost}_{\text{ElasticNet}} = \sum (y_i - \hat{y}_i)^2 + \alpha \left[ \rho \sum |\beta_j| + \frac{1 - \rho}{2} \sum \beta_j^2 \right]$$

where $\rho$ (`l1_ratio` in sklearn) controls the mix:

- $\rho = 1$: pure Lasso.
- $\rho = 0$: pure Ridge.
- $0 < \rho < 1$: a blend of both.

**When to use ElasticNet:** when you have many correlated features. Lasso tends to
pick one feature from a correlated group and discard the rest; ElasticNet spreads
the weight more evenly among correlated features.

In [ ]:
pipe_enet = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
])
pipe_enet.fit(X_train, y_train)

enet_coefs = pipe_enet.named_steps['reg'].coef_
n_zero_enet = np.sum(np.abs(enet_coefs) < 1e-10)

print(f'ElasticNet (alpha=1.0, l1_ratio=0.5, degree 10):')
print(f'  Train R²: {pipe_enet.score(X_train, y_train):.4f}')
print(f'  Test R²:  {pipe_enet.score(X_test, y_test):.4f}')
print(f'  Coefficients zeroed out: {n_zero_enet} / {len(enet_coefs)}')

---
## 8. Comparing Ridge, Lasso, and OLS Side by Side

In [ ]:
x_vis = np.linspace(-3.5, 3.5, 400).reshape(-1, 1)

models = {
    'OLS (no penalty)': pipe_ols,
    'Ridge (alpha=1)': pipe_ridge,
    'Lasso (alpha=1)': pipe_lasso,
    'ElasticNet (alpha=1)': pipe_enet,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (name, model) in zip(axes, models.items()):
    y_vis = model.predict(x_vis)
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    
    ax.scatter(X_train, y_train, alpha=0.4, edgecolors='k', linewidths=0.3, label='Train')
    ax.scatter(X_test, y_test, alpha=0.4, marker='s', edgecolors='k', linewidths=0.3, label='Test')
    ax.plot(x_vis, y_vis, 'r-', linewidth=2)
    ax.set_title(f'{name}\nTrain R²={train_r2:.3f}, Test R²={test_r2:.3f}')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(y.min() - 15, y.max() + 15)
    ax.legend(fontsize=8)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Degree-10 Polynomial: OLS vs Regularized Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

OLS produces wild oscillations. The regularized models tame the curve, producing
much better generalization (higher test R²).

---
## 9. The Alpha Hyperparameter — How to Choose It

Alpha controls the regularization strength. Too low = overfitting; too high =
underfitting. We need a principled way to pick alpha.

### 9.1 Test R² vs Alpha

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(alphas, ridge_train_r2, 'b--', label='Ridge Train R²', alpha=0.7)
plt.plot(alphas, ridge_test_r2, 'b-', label='Ridge Test R²', linewidth=2)
plt.plot(alphas, lasso_test_r2, 'r-', label='Lasso Test R²', linewidth=2)
plt.xscale('log')
plt.xlabel(r'$\alpha$')
plt.ylabel('R² Score')
plt.title(r'R² vs $\alpha$ — Finding the Sweet Spot')
plt.legend()
plt.show()

The test R² peaks at an intermediate alpha — not too small (overfitting) and
not too large (underfitting). But evaluating on a single test set is noisy.
**Cross-validation** gives a more reliable estimate.

---
## 10. Cross-Validation for Model Selection

### 10.1 What Is Cross-Validation?

Instead of a single train/test split, **k-fold cross-validation** divides the
training data into $k$ equal folds and runs $k$ experiments:

1. For each fold $i$, train on all folds except $i$ and validate on fold $i$.
2. Average the $k$ validation scores.

This gives a more robust estimate of model performance because every data point
is used for validation exactly once.

### 10.2 cross_val_score

In [ ]:
# 5-fold CV on the Ridge pipeline
pipe_ridge_cv = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0))
])

cv_scores = cross_val_score(pipe_ridge_cv, X_train, y_train, cv=5, scoring='r2')

print(f'5-Fold CV R² scores: {cv_scores.round(4)}')
print(f'Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

The mean CV R² gives a single number summarising expected performance on unseen
data, and the standard deviation indicates how stable that estimate is.

### 10.3 CV Across Multiple Alpha Values

In [ ]:
alpha_candidates = np.logspace(-2, 4, 30)
ridge_cv_means = []
ridge_cv_stds = []

for a in alpha_candidates:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', Ridge(alpha=a))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    ridge_cv_means.append(scores.mean())
    ridge_cv_stds.append(scores.std())

ridge_cv_means = np.array(ridge_cv_means)
ridge_cv_stds = np.array(ridge_cv_stds)

best_idx = np.argmax(ridge_cv_means)
best_alpha = alpha_candidates[best_idx]

plt.figure(figsize=(10, 5))
plt.plot(alpha_candidates, ridge_cv_means, 'b-o', markersize=4)
plt.fill_between(alpha_candidates,
                 ridge_cv_means - ridge_cv_stds,
                 ridge_cv_means + ridge_cv_stds,
                 alpha=0.2)
plt.axvline(best_alpha, color='red', linestyle='--',
            label=f'Best alpha = {best_alpha:.3f}')
plt.xscale('log')
plt.xlabel(r'$\alpha$')
plt.ylabel('Mean CV R²')
plt.title('Ridge: Cross-Validated R² vs Alpha')
plt.legend()
plt.show()

print(f'Best alpha: {best_alpha:.4f}, Mean CV R²: {ridge_cv_means[best_idx]:.4f}')

---
## 11. GridSearchCV — Automating Hyperparameter Search

`GridSearchCV` automates the loop above and handles the bookkeeping for you.
It fits a model for every combination of hyperparameters, evaluates each with
cross-validation, and selects the best.

In [ ]:
pipe_grid = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge())
])

param_grid = {
    'poly__degree': [3, 5, 7, 10],
    'reg__alpha': np.logspace(-2, 4, 15)
}

grid_search = GridSearchCV(
    pipe_grid, param_grid, cv=5, scoring='r2', return_train_score=True
)
grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV R²:      {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate the best model on the held-out test set
best_model = grid_search.best_estimator_
test_r2 = best_model.score(X_test, y_test)

print(f'Test R² of best model: {test_r2:.4f}')

### 11.1 Visualising the Grid Search Results

In [ ]:
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)

fig, ax = plt.subplots(figsize=(10, 6))
for degree in param_grid['poly__degree']:
    mask = results['param_poly__degree'] == degree
    subset = results[mask]
    ax.plot(subset['param_reg__alpha'].astype(float),
            subset['mean_test_score'],
            'o-', label=f'Degree {degree}')

ax.set_xscale('log')
ax.set_xlabel(r'Ridge $\alpha$')
ax.set_ylabel('Mean CV R²')
ax.set_title('GridSearchCV Results: Degree vs Alpha')
ax.legend()
plt.show()

---
## 12. GridSearchCV with Lasso

In [ ]:
pipe_lasso_grid = Pipeline([
    ('poly', PolynomialFeatures(degree=10, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Lasso(max_iter=10000))
])

param_grid_lasso = {
    'reg__alpha': np.logspace(-3, 2, 30)
}

grid_lasso = GridSearchCV(
    pipe_lasso_grid, param_grid_lasso, cv=5, scoring='r2'
)
grid_lasso.fit(X_train, y_train)

print(f'Lasso best alpha: {grid_lasso.best_params_["reg__alpha"]:.4f}')
print(f'Lasso best CV R²: {grid_lasso.best_score_:.4f}')
print(f'Lasso test R²:    {grid_lasso.best_estimator_.score(X_test, y_test):.4f}')

best_lasso_coefs = grid_lasso.best_estimator_.named_steps['reg'].coef_
n_zero_best = np.sum(np.abs(best_lasso_coefs) < 1e-10)
print(f'Zeroed coefficients: {n_zero_best} / {len(best_lasso_coefs)}')

---
## 13. Learning Curves — Diagnosing Bias and Variance

A **learning curve** plots the training and validation scores as a function of
the training set size. It reveals:

- **High bias (underfitting)**: both scores are low and converge — more data will
  not help; the model is too simple.
- **High variance (overfitting)**: training score is high but validation score is
  low — there is a big gap; more data may help.

In [ ]:
def plot_learning_curve(estimator, title, X, y, cv=5):
    """Plot a learning curve for the given estimator."""
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='r2', n_jobs=-1
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
    plt.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation score')
    plt.xlabel('Training set size')
    plt.ylabel('R² Score')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.ylim(-0.5, 1.1)

In [ ]:
# Generate a larger dataset for learning curve analysis
n_lc = 300
x_lc = rng.uniform(-3, 3, size=n_lc)
y_lc = 0.5 * x_lc**3 - x_lc**2 + 2 * x_lc + 5 + rng.normal(0, 3, size=n_lc)
X_lc = x_lc.reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Underfitting model (degree 1)
plt.sca(axes[0])
pipe_underfit = Pipeline([
    ('poly', PolynomialFeatures(degree=1, include_bias=False)),
    ('reg', LinearRegression())
])
plot_learning_curve(pipe_underfit, 'Underfitting (Degree 1)', X_lc, y_lc)

# Good fit (degree 3)
plt.sca(axes[1])
pipe_good = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('reg', LinearRegression())
])
plot_learning_curve(pipe_good, 'Good Fit (Degree 3)', X_lc, y_lc)

# Overfitting model (degree 15, no regularization)
plt.sca(axes[2])
pipe_overfit = Pipeline([
    ('poly', PolynomialFeatures(degree=15, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])
plot_learning_curve(pipe_overfit, 'Overfitting (Degree 15)', X_lc, y_lc)

plt.tight_layout()
plt.show()

**Reading the plots:**

- **Left (underfitting)**: both curves converge to a low score. The model is too
  simple — adding data will not help.
- **Center (good fit)**: the curves converge to a high score with a small gap.
- **Right (overfitting)**: training score is near-perfect but validation lags far
  behind. More data or regularization would help.

### 13.1 Learning Curve: Effect of Regularization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Degree 15 without regularization
plt.sca(axes[0])
plot_learning_curve(pipe_overfit, 'Degree 15, No Regularization', X_lc, y_lc)

# Degree 15 with Ridge regularization
plt.sca(axes[1])
pipe_ridge_lc = Pipeline([
    ('poly', PolynomialFeatures(degree=15, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=10.0))
])
plot_learning_curve(pipe_ridge_lc, 'Degree 15, Ridge (alpha=10)', X_lc, y_lc)

plt.tight_layout()
plt.show()

Ridge narrows the gap between training and validation curves — the model
generalises much better with regularization applied.

---
## 14. Comparing All Models on the Same Dataset

In [ ]:
comparison_models = {
    'Linear (degree 1)': Pipeline([
        ('poly', PolynomialFeatures(degree=1, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ]),
    'OLS (degree 10)': Pipeline([
        ('poly', PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ]),
    'Ridge (degree 10)': Pipeline([
        ('poly', PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', Ridge(alpha=grid_search.best_params_.get('reg__alpha', 1.0)))
    ]),
    'Lasso (degree 10)': Pipeline([
        ('poly', PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', Lasso(alpha=grid_lasso.best_params_['reg__alpha'], max_iter=10000))
    ]),
    'ElasticNet (degree 10)': Pipeline([
        ('poly', PolynomialFeatures(degree=10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
    ]),
}

print(f'{"Model":<25} {"Train R²":>10} {"CV R² (mean)":>14} {"Test R²":>10}')
print('-' * 62)

for name, model in comparison_models.items():
    model.fit(X_train, y_train)
    train_r2 = model.score(X_train, y_train)
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    test_r2 = model.score(X_test, y_test)
    print(f'{name:<25} {train_r2:>10.4f} {cv_r2:>14.4f} {test_r2:>10.4f}')

The regularized models (Ridge, Lasso, ElasticNet) consistently achieve better
test performance than unpenalised OLS when the model has excess capacity.

---
## 15. Practical Workflow: Scale, Split, Regularize, CV, Evaluate

Putting it all together, a typical regression workflow looks like this:

1. **Split** the data into train and test sets (hold the test set aside until the end).
2. **Build a pipeline**: polynomial features (if needed) + scaler + regularized model.
3. **Tune hyperparameters** with `GridSearchCV` on the training set.
4. **Evaluate** the best model on the held-out test set.

### 15.1 End-to-End Example

In [ ]:
# Step 1: Generate and split data
n_final = 200
x_final = rng.uniform(-4, 4, size=n_final)
y_final = 1.5 * x_final**2 - 0.5 * x_final + 3 + rng.normal(0, 4, size=n_final)
X_final = x_final.reshape(-1, 1)

X_tr, X_te, y_tr, y_te = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

print(f'Train: {X_tr.shape[0]} samples, Test: {X_te.shape[0]} samples')

In [ ]:
# Step 2: Define pipeline
pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', Ridge())
])

# Step 3: Grid search
params = {
    'poly__degree': [2, 3, 4, 5, 7],
    'reg__alpha': np.logspace(-2, 3, 20)
}

search = GridSearchCV(pipeline, params, cv=5, scoring='r2', return_train_score=True)
search.fit(X_tr, y_tr)

print(f'Best parameters: {search.best_params_}')
print(f'Best CV R²:      {search.best_score_:.4f}')

In [ ]:
# Step 4: Final evaluation on the test set
final_model = search.best_estimator_
final_r2 = final_model.score(X_te, y_te)
final_rmse = np.sqrt(mean_squared_error(y_te, final_model.predict(X_te)))

print(f'Final test R²:   {final_r2:.4f}')
print(f'Final test RMSE: {final_rmse:.4f}')

In [ ]:
# Visualise the final model
x_vis_final = np.linspace(-4.5, 4.5, 400).reshape(-1, 1)
y_vis_final = final_model.predict(x_vis_final)

plt.scatter(X_tr, y_tr, alpha=0.4, label='Train', edgecolors='k', linewidths=0.3)
plt.scatter(X_te, y_te, alpha=0.4, marker='s', label='Test', edgecolors='k', linewidths=0.3)
plt.plot(x_vis_final, y_vis_final, 'r-', linewidth=2, label='Best model')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Final Model: Degree={search.best_params_["poly__degree"]}, '
          f'Alpha={search.best_params_["reg__alpha"]:.3f}, '
          f'Test R²={final_r2:.3f}')
plt.legend()
plt.show()

---
## 16. Scaling Matters for Regularization

**Important:** Ridge and Lasso penalise coefficients by their magnitude.
If features are on different scales, features with larger values will have
artificially smaller coefficients, and the penalty will not treat all features
equally.

**Always standardise features** (zero mean, unit variance) before applying
regularization. `StandardScaler` in the pipeline handles this.

In [ ]:
# Demonstration: Ridge with and without scaling
# Create features on very different scales
X_scale_demo = np.column_stack([
    rng.uniform(0, 1, size=200),       # x1: small scale
    rng.uniform(0, 1000, size=200),     # x2: large scale
])
y_scale_demo = 5 * X_scale_demo[:, 0] + 0.005 * X_scale_demo[:, 1] + rng.normal(0, 0.5, size=200)

# Without scaling
ridge_no_scale = Ridge(alpha=1.0).fit(X_scale_demo, y_scale_demo)
print(f'Without scaling — coefs: {ridge_no_scale.coef_.round(6)}')

# With scaling
pipe_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0))
])
pipe_scaled.fit(X_scale_demo, y_scale_demo)
print(f'With scaling    — coefs: {pipe_scaled.named_steps["reg"].coef_.round(6)}')
print(f'\nThe scaled coefficients are comparable in magnitude, so the penalty')
print(f'is applied fairly across both features.')

---
## 17. Summary

| Concept | Key Takeaway |
|---------|-------------|
| Regularization | Adds a penalty to the cost function to prevent overfitting |
| Ridge (L2) | Shrinks all coefficients toward zero; keeps all features |
| Lasso (L1) | Can zero out coefficients; performs feature selection |
| ElasticNet | Combines L1 and L2; good for correlated features |
| Alpha | Controls penalty strength; chosen by cross-validation |
| Cross-validation | Robust estimate of model performance on unseen data |
| GridSearchCV | Automates hyperparameter search with CV |
| Learning curves | Diagnose bias (underfitting) vs variance (overfitting) |
| Feature scaling | Essential before regularization — use StandardScaler |
| Practical workflow | Split, pipeline (poly + scale + regularize), GridSearchCV, evaluate |